# Experiment: CatBoost on the full training set

CatBoost with categorical `status`/`request_method` fitted on all 54,480 training logs, first test prediction and a first look at post-evaluation rules.

In [ ]:
DATA_DIR = "../data"
SUBMISSION_DIR = "../data/submissions"

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score

import catboost as cb
# import lightgbm as lgb
# import xgboost as xgb

import tensorflow as tf
from tensorflow.keras import backend as K

## Load Data

In [57]:
ori_train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
ori_train_df.head()

,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic
0,0,"Aug 29, 2023 @ 17:00:03.058",114.4.220.184,-,vm-director-nginx,30/Aug/2023:00:00:02 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306444,2,114.4.220.184,192.168.1.99,"[""non-attack""]"
1,1,"Aug 29, 2023 @ 17:00:33.070",10.200.160.148,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0,0.002,0.002,10.45.184.154:32442,404,688306865,2,10.200.160.148,192.168.1.99,"[""non-attack""]"
2,2,"Aug 29, 2023 @ 17:00:33.072",180.248.46.18,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306855,2,180.248.46.18,192.168.1.99,"[""non-attack""]"
3,3,"Aug 29, 2023 @ 17:00:33.076",125.166.3.1,-,vm-director-nginx,30/Aug/2023:00:00:32 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0,0.002,0.002,10.45.184.154:32442,404,688306869,2,125.166.3.1,192.168.1.99,"[""non-attack""]"
4,4,"Aug 29, 2023 @ 17:00:59.118",103.108.23.13,-,vm-director-nginx,30/Aug/2023:00:00:58 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.003,0.003,10.45.184.154:32442,404,688307225,2,103.108.23.13,192.168.1.99,"[""non-attack""]"


In [58]:
ori_train_df

,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic
0,0,"Aug 29, 2023 @ 17:00:03.058",114.4.220.184,-,vm-director-nginx,30/Aug/2023:00:00:02 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306444,2,114.4.220.184,192.168.1.99,"[""non-attack""]"
1,1,"Aug 29, 2023 @ 17:00:33.070",10.200.160.148,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0,0.002,0.002,10.45.184.154:32442,404,688306865,2,10.200.160.148,192.168.1.99,"[""non-attack""]"
2,2,"Aug 29, 2023 @ 17:00:33.072",180.248.46.18,-,vm-director-nginx,30/Aug/2023:00:00:31 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.148:32442,404,688306855,2,180.248.46.18,192.168.1.99,"[""non-attack""]"
3,3,"Aug 29, 2023 @ 17:00:33.076",125.166.3.1,-,vm-director-nginx,30/Aug/2023:00:00:32 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0,0.002,0.002,10.45.184.154:32442,404,688306869,2,125.166.3.1,192.168.1.99,"[""non-attack""]"
4,4,"Aug 29, 2023 @ 17:00:59.118",103.108.23.13,-,vm-director-nginx,30/Aug/2023:00:00:58 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.003,0.003,10.45.184.154:32442,404,688307225,2,103.108.23.13,192.168.1.99,"[""non-attack""]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54475,54475,"Sep 5, 2023 @ 00:26:40.840",202.80.216.211,-,vm-director-nginx,05/Sep/2023:07:26:39 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0,0.002,0.002,10.45.184.154:32442,404,721724820,2,202.80.216.211,192.168.1.99,"[""non-attack""]"
54476,54476,"Sep 5, 2023 @ 00:26:40.880",114.5.245.55,-,vm-director-nginx,05/Sep/2023:07:26:40 +0700,GET /.well-known/assetlinks.json HTTP/2.0,404,74,-,...,0,0.001,0.001,10.45.184.154:32442,404,721725022,1,114.5.245.55,192.168.1.99,"[""non-attack""]"
54477,54477,"Sep 5, 2023 @ 00:26:40.907",114.5.245.55,-,vm-director-nginx,05/Sep/2023:07:26:40 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.154:32442,404,721725023,2,114.5.245.55,192.168.1.99,"[""non-attack""]"
54478,54478,"Sep 5, 2023 @ 00:26:40.913",10.27.97.71,-,vm-director-nginx,05/Sep/2023:07:26:40 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.002,0.002,10.45.184.148:32442,404,721725017,2,10.27.97.71,192.168.1.99,"[""non-attack""]"


In [59]:
ori_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54480 entries, 0 to 54479
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      54480 non-null  int64  
 1   timestamp               54480 non-null  object 
 2   remote_addr             54480 non-null  object 
 3   remote_user             54480 non-null  object 
 4   hostname                54480 non-null  object 
 5   time_local              54480 non-null  object 
 6   request                 54480 non-null  object 
 7   status                  54480 non-null  int64  
 8   body_bytes_sent         54480 non-null  int64  
 9   http_referer            54480 non-null  object 
 10  http_user_agent         54480 non-null  object 
 11  http_x_forwarded_for    54480 non-null  object 
 12  time_iso8601            54480 non-null  object 
 13  request_method          54480 non-null  object 
 14  request_length          54480 non-null

## Feature Engineering

In [60]:
ori_train_df.upstream_connect_time.value_counts()

0               27741
0.001           25571
0.002             526
-, 0.000           88
-                  87
-, -, -            70
0.004              66
0.003              64
0.005              49
0.008              36
0.006              33
0.007              20
0.009              14
0.012              12
0.01               11
0.018               9
0.026               6
0.02                5
0.013               5
0.016               5
0.017               4
0.011               4
0.014               4
0.021               4
-, 0.001            4
0.046               3
0.036               3
0.027               3
0.037               2
0.025               2
0.019               1
0.031               1
0.092               1
1.003               1
-, 0.013            1
-, 0.002            1
0.068               1
0.001, 0.001        1
0.032               1
0.034               1
0.067               1
0.489               1
0.04                1
0.035               1
0.265               1
0.169     

In [8]:
ori_train_df.upstream_response_time.value_counts()

0.002    1339
0.001     817
0.003     448
0          98
0.004      48
         ... 
0.199       1
0.587       1
0.286       1
0.893       1
0.674       1
Name: upstream_response_time, Length: 601, dtype: int64

In [64]:
def clean_upstream_connect_time(x):
    x = x.split(', ')
    for i in range(len(x)):
        try:
            x[i] = float(x[i])
        except ValueError:
            x[i] = 0
    return np.max(x)

def extract_domain_from_url(url):
    pattern = r'^(?:https?:\/\/)?(?:[^@\/\n]+@)?(?:www\.)?([^:\/\n]+)'
    match = re.search(pattern, url)

    if match:
        domain = match.group(1)
        return domain
    else:
        return None
    
def group_http_referer(x):
    x = extract_domain_from_url(x)
    if "notrealdomain" in x:
        return 'notrealdomain'
    elif "175" in x:
        return 'ipaddress'
    elif ".com" in x or '.co.id' in x:
        return 'realwebsite'
    else:
        return x

train_df = ori_train_df.copy()
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
train_df['upstream_connect_time'] = train_df['upstream_connect_time'].apply(clean_upstream_connect_time)
train_df['upstream_header_time'] = train_df['upstream_header_time'].apply(clean_upstream_connect_time)
train_df['upstream_response_time'] = train_df['upstream_response_time'].apply(clean_upstream_connect_time)
train_df['http_referer'] = train_df['http_referer'].apply(group_http_referer)
# train_df.drop(train_df[train_df['rule.mitre.tactic'] == '["Credential Access"]'].index, inplace=True)
train_df.drop([
    'id',
    # 'Unnamed: 0',
    'remote_addr',
    'remote_user',
    'hostname',
    'time_local',
    'request',
    'http_referer',
    'http_referer2',
    'http_x_forwarded_for',
    'http_user_agent',
    'time_iso8601',
    'upstream_addr',
    'upstream_status',
    # 'connection',
    'data.srcip',
    'agent.ip',
    # 'pairwise'
], axis=1, inplace=True)
cat_cols = ['status', 'request_method', 'http_referer']

In [65]:
cat_cols = ['status', 'request_method']

In [66]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54480 entries, 0 to 54479
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               54480 non-null  datetime64[ns]
 1   status                  54480 non-null  int64         
 2   body_bytes_sent         54480 non-null  int64         
 3   request_method          54480 non-null  object        
 4   request_length          54480 non-null  int64         
 5   bytes_sent              54480 non-null  int64         
 6   request_time            54480 non-null  float64       
 7   upstream_connect_time   54480 non-null  float64       
 8   upstream_header_time    54480 non-null  float64       
 9   upstream_response_time  54480 non-null  float64       
 10  connection              54480 non-null  int64         
 11  connection_requests     54480 non-null  int64         
 12  rule.mitre.tactic       54480 non-null  object

In [67]:
train_df.head()

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,rule.mitre.tactic
0,2023-08-29 17:00:03.058,404,118,GET,79,330,0.002,0.001,0.002,0.002,688306444,2,"[""non-attack""]"
1,2023-08-29 17:00:33.070,404,118,GET,79,330,0.002,0.000,0.002,0.002,688306865,2,"[""non-attack""]"
2,2023-08-29 17:00:33.072,404,118,GET,92,330,0.003,0.001,0.002,0.002,688306855,2,"[""non-attack""]"
3,2023-08-29 17:00:33.076,404,118,GET,81,330,0.002,0.000,0.002,0.002,688306869,2,"[""non-attack""]"
4,2023-08-29 17:00:59.118,404,118,GET,82,330,0.003,0.001,0.003,0.003,688307225,2,"[""non-attack""]"


In [68]:
train_df.upstream_connect_time.value_counts()

0.000    27986
0.001    25577
0.002      527
0.004       66
0.003       64
0.005       49
0.008       36
0.006       33
0.007       20
0.009       14
0.012       12
0.010       11
0.018        9
0.013        6
0.026        6
0.016        5
0.020        5
0.017        4
0.011        4
0.014        4
0.021        4
0.046        3
0.036        3
0.027        3
0.025        2
0.037        2
0.067        1
0.040        1
0.019        1
0.034        1
0.031        1
0.092        1
0.068        1
0.032        1
0.169        1
1.003        1
0.489        1
0.265        1
0.035        1
0.108        1
0.060        1
0.045        1
0.053        1
0.033        1
0.015        1
0.118        1
0.029        1
0.126        1
0.024        1
0.061        1
0.064        1
Name: upstream_connect_time, dtype: int64

## Modeling

In [69]:
X = train_df.drop('rule.mitre.tactic', axis=1)
y = train_df['rule.mitre.tactic']

In [70]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54480 entries, 0 to 54479
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               54480 non-null  datetime64[ns]
 1   status                  54480 non-null  int64         
 2   body_bytes_sent         54480 non-null  int64         
 3   request_method          54480 non-null  object        
 4   request_length          54480 non-null  int64         
 5   bytes_sent              54480 non-null  int64         
 6   request_time            54480 non-null  float64       
 7   upstream_connect_time   54480 non-null  float64       
 8   upstream_header_time    54480 non-null  float64       
 9   upstream_response_time  54480 non-null  float64       
 10  connection              54480 non-null  int64         
 11  connection_requests     54480 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(6), ob

In [21]:
cbc = cb.CatBoostClassifier(task_type="GPU",
                            cat_features=cat_cols,
                            verbose=1, 
                            random_seed=42)
cbc.fit(X, y)

Learning rate set to 0.089223
0:	learn: 1.4207899	total: 25.7ms	remaining: 25.7s
1:	learn: 1.1887284	total: 48.3ms	remaining: 24.1s
2:	learn: 1.0161758	total: 82.8ms	remaining: 27.5s
3:	learn: 0.8845763	total: 110ms	remaining: 27.3s
4:	learn: 0.7788186	total: 133ms	remaining: 26.5s
5:	learn: 0.6904524	total: 157ms	remaining: 26s
6:	learn: 0.6178754	total: 187ms	remaining: 26.6s
7:	learn: 0.5559784	total: 209ms	remaining: 25.9s
8:	learn: 0.5015384	total: 234ms	remaining: 25.7s
9:	learn: 0.4547264	total: 258ms	remaining: 25.6s
10:	learn: 0.4134524	total: 285ms	remaining: 25.6s
11:	learn: 0.3774042	total: 306ms	remaining: 25.2s
12:	learn: 0.3449731	total: 336ms	remaining: 25.5s
13:	learn: 0.3165893	total: 368ms	remaining: 25.9s
14:	learn: 0.2910987	total: 395ms	remaining: 26s
15:	learn: 0.2684724	total: 422ms	remaining: 26s
16:	learn: 0.2479705	total: 451ms	remaining: 26.1s
17:	learn: 0.2293399	total: 477ms	remaining: 26s
18:	learn: 0.2124724	total: 566ms	remaining: 29.2s
19:	learn: 0.197

## Submission

In [71]:
ori_test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
ori_test_df.head()

,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
0,54481,"Sep 5, 2023 @ 00:26:44.850",10.26.152.242,-,vm-director-nginx,05/Sep/2023:07:26:43 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.000,0.001,0.001,10.45.184.154:32442,404,721725502,2,10.26.152.242,192.168.1.99
1,54482,"Sep 5, 2023 @ 00:26:44.880",202.80.215.96,-,vm-director-nginx,05/Sep/2023:07:26:44 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.001,0.003,0.003,10.45.184.148:32442,404,721725753,2,202.80.215.96,192.168.1.99
2,54483,"Sep 5, 2023 @ 00:26:46.898",36.85.73.24,-,vm-director-nginx,05/Sep/2023:07:26:44 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.003,0.001,0.002,0.002,10.45.184.148:32442,404,721725865,2,36.85.73.24,192.168.1.99
3,54484,"Sep 5, 2023 @ 00:26:46.945",180.248.23.157,-,vm-director-nginx,05/Sep/2023:07:26:46 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.000,0.002,0.002,10.45.184.148:32442,404,721725740,2,180.248.23.157,192.168.1.99
4,54485,"Sep 5, 2023 @ 00:26:48.856",182.1.80.255,-,vm-director-nginx,05/Sep/2023:07:26:46 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.000,0.001,0.001,10.45.184.154:32442,404,721726205,2,182.1.80.255,192.168.1.99


In [72]:
ori_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13599 entries, 0 to 13598
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      13599 non-null  int64  
 1   timestamp               13599 non-null  object 
 2   remote_addr             13599 non-null  object 
 3   remote_user             13599 non-null  object 
 4   hostname                13599 non-null  object 
 5   time_local              13599 non-null  object 
 6   request                 13599 non-null  object 
 7   status                  13599 non-null  int64  
 8   body_bytes_sent         13599 non-null  int64  
 9   http_referer            13599 non-null  object 
 10  http_user_agent         13599 non-null  object 
 11  http_x_forwarded_for    13599 non-null  object 
 12  time_iso8601            13599 non-null  object 
 13  request_method          13599 non-null  object 
 14  request_length          13599 non-null

In [73]:
train_df.status.value_counts()

404    51632
200     1777
400      487
502      158
403      151
500      116
504       51
407       50
422       42
303        8
302        4
405        2
301        1
413        1
Name: status, dtype: int64

In [25]:
ori_test_df.status.value_counts()

404    12722
200      656
400      152
422       25
502       22
500       11
504        6
416        2
403        1
405        1
407        1
Name: status, dtype: int64

In [74]:
test_df = ori_test_df.copy()
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
# test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])
test_df['upstream_connect_time'] = test_df['upstream_connect_time'].apply(clean_upstream_connect_time)
test_df['upstream_header_time'] = test_df['upstream_header_time'].apply(clean_upstream_connect_time)
test_df['upstream_response_time'] = test_df['upstream_response_time'].apply(clean_upstream_connect_time)
# test_df['http_referer'] = test_df['http_referer'].apply(group_http_referer)
test_df['hour'] = test_df['timestamp'].dt.hour
test_df.drop([
    'id',
    # 'Unnamed: 0',
    'remote_addr',
    'remote_user',
    'hostname',
    'time_local',
    'request',
    'http_referer',
    'http_referer2',
    'http_x_forwarded_for',
    'http_user_agent',
    'time_iso8601',
    'upstream_addr',
    'upstream_status',
    # 'connection',
    'data.srcip',
    'agent.ip',
    # 'pairwise'
], axis=1, inplace=True)
test_df.head()

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,hour
0,2023-09-05 00:26:44.850,404,118,GET,80,330,0.002,0.000,0.001,0.001,721725502,2,0
1,2023-09-05 00:26:44.880,404,118,GET,83,330,0.002,0.001,0.003,0.003,721725753,2,0
2,2023-09-05 00:26:46.898,404,118,GET,85,330,0.003,0.001,0.002,0.002,721725865,2,0
3,2023-09-05 00:26:46.945,404,118,GET,80,330,0.002,0.000,0.002,0.002,721725740,2,0
4,2023-09-05 00:26:48.856,404,118,GET,84,330,0.001,0.000,0.001,0.001,721726205,2,0


In [75]:
y_test_pred = cbc.predict(test_df)
y_test_pred

array([['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ...,
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]']], dtype=object)

In [76]:
y_test_pred[test_df[test_df.status != 404].index]

array([['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Reconnaissance"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Impact"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Initial Access"]'],
       ['["Reconnaissance"]'],
       ['["Impact"]'],
       ['["Reconnaissance"]'],
       ['

In [31]:
test_df[test_df.status != 404].head(10)

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,hour
43,2023-09-05 00:28:43.454,200,228,POST,154,412,0.035,0.001,0.035,0.035,721742964,139,0
45,2023-09-05 00:28:45.237,200,2274,POST,620,2515,0.279,0.001,0.273,0.279,721742545,15,0
54,2023-09-05 00:28:35.301,200,102,POST,288,333,0.063,0.001,0.061,0.061,721742545,7,0
58,2023-09-05 00:28:13.486,200,5722,POST,302,5931,0.139,0.000,0.137,0.138,721737354,174,0
78,2023-09-05 00:28:55.139,200,167,POST,469,351,0.078,0.001,0.047,0.047,721742997,173,0
98,2023-09-05 00:29:07.108,400,78,POST,1380,473,0.005,0.001,0.006,0.006,721747503,1,0
113,2023-09-05 00:30:07.838,200,100,POST,325,284,0.040,0.000,0.037,0.037,721755939,175,0
148,2023-09-05 00:30:19.748,200,27,POST,270,210,0.159,0.000,0.088,0.088,721763354,45,0
150,2023-09-05 00:30:53.567,200,27,POST,160,210,0.066,0.001,0.066,0.066,721749840,211,0
152,2023-09-05 00:30:55.409,200,166,POST,470,350,0.093,0.001,0.038,0.038,721767373,185,0


In [77]:
submission_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
submission_df['rule.mitre.tactic'] = y_test_pred
submission_df.head()

,id,rule.mitre.tactic
0,54481,"[""Reconnaissance""]"
1,54482,"[""Reconnaissance""]"
2,54483,"[""Reconnaissance""]"
3,54484,"[""Reconnaissance""]"
4,54485,"[""Reconnaissance""]"


In [78]:
submission_df['rule.mitre.tactic'].value_counts()

["Reconnaissance"]     12674
["Impact"]               753
["Defense Evasion"]      160
["Initial Access"]         8
["non-attack"]             4
Name: rule.mitre.tactic, dtype: int64

In [41]:
train_full = pd.read_csv(f"{DATA_DIR}/train.csv")

In [44]:
cross_tab = pd.crosstab(train_full['http_referer'], train_full['rule.mitre.tactic'])
# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]

# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

In [47]:
ori_test_df[ori_test_df['http_referer'].isin(non_attack_http_list)]


,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
257,54738,"Sep 5, 2023 @ 00:31:47.210",10.200.144.115,-,vm-director-nginx,05/Sep/2023:07:31:46 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/open.php,...,0.001,0.001,0.001,0.001,10.45.184.59:8001,404,721783562,3,10.200.144.115,192.168.1.99
521,55002,"Sep 5, 2023 @ 00:35:55.517",36.71.145.232,-,vm-director-nginx,05/Sep/2023:07:35:55 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/kb/faq.ph...,...,0.001,0.001,0.002,0.002,10.45.184.59:8001,404,721825599,3,36.71.145.232,192.168.1.99
1400,55881,"Sep 5, 2023 @ 00:55:12.999",10.200.160.31,-,vm-director-nginx,05/Sep/2023:07:55:12 +0700,GET /ujian.php?key=MzA7MTk7MDA7MDI7MTsyMDIzOzE...,404,36,https://siam.notrealdomain.com/exam.php,...,0.002,0.000,0.000,0.000,175.45.184.244:80,404,721970221,20,10.200.160.31,192.168.1.99
1419,55900,"Sep 5, 2023 @ 00:55:02.907",10.200.160.31,-,vm-director-nginx,05/Sep/2023:07:55:01 +0700,GET /ujian.php?key=MzA7MTk7MDA7MDI7MTsyMDIzOzE...,404,36,https://siam.notrealdomain.com/exam.php,...,0.002,0.000,0.001,0.001,175.45.184.244:80,404,721970221,19,10.200.160.31,192.168.1.99
1486,55967,"Sep 5, 2023 @ 00:57:00.966",10.34.3.151,-,vm-director-nginx,05/Sep/2023:07:57:00 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/,...,0.001,0.001,0.002,0.002,10.45.184.59:8001,404,721986851,2,10.34.3.151,192.168.1.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13201,67682,"Sep 6, 2023 @ 02:03:57.362",118.99.112.12,-,vm-director-nginx,06/Sep/2023:09:03:56 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2425,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.011,0.000,0.010,0.010,175.45.185.12:80,404,729170418,40,118.99.112.12,192.168.1.99
13438,67919,"Sep 6, 2023 @ 02:09:55.723",182.1.67.189,-,vm-director-nginx,06/Sep/2023:09:09:54 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2421,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.012,0.000,0.011,0.011,175.45.185.12:80,404,729216416,41,182.1.67.189,192.168.1.99
13483,67964,"Sep 6, 2023 @ 02:10:59.831",110.138.213.129,-,vm-director-nginx,06/Sep/2023:09:10:59 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2424,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.012,0.000,0.012,0.012,175.45.185.12:80,404,729237155,16,110.138.213.129,192.168.1.99
13493,67974,"Sep 6, 2023 @ 02:11:03.777",182.1.67.189,-,vm-director-nginx,06/Sep/2023:09:11:02 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2423,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.015,0.001,0.014,0.015,175.45.185.12:80,404,729216416,54,182.1.67.189,192.168.1.99


In [54]:
ori_test_df[ori_test_df['request_method'].isin(["HEAD"])] 


,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
192,54673,"Sep 5, 2023 @ 00:31:19.288",180.248.43.56,-,vm-director-nginx,05/Sep/2023:07:31:17 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.002,0.001,0.002,0.002,10.45.184.154:32442,404,721779079,1,180.248.43.56,192.168.1.99
6027,60508,"Sep 5, 2023 @ 22:34:53.144",202.80.214.69,-,vm-director-nginx,06/Sep/2023:05:34:52 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.002,0.000,0.001,0.001,10.45.184.154:32442,404,728024745,1,202.80.214.69,192.168.1.99
8725,63206,"Sep 6, 2023 @ 00:21:34.290",10.200.145.39,-,vm-director-nginx,06/Sep/2023:07:21:32 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.001,0.000,0.001,0.001,10.45.184.154:32442,404,728392396,1,10.200.145.39,192.168.1.99


In [34]:
submission_df.to_csv(f"{SUBMISSION_DIR}/submission1.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'submissions'

## Post Evaluation

In [ ]:

# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

test[test['http_referer'].isin(non_attack_http_list)] = ["non-attack"]
